In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Definition} $$

$$ 
z = xw + b
$$

$$ \text{Derivative} $$

$$ \frac{dz}{dw} = x $$

$$ \frac{dz}{db} = 1 $$

$$ \text{Jacobian} $$

$$
J_z(w) =
\begin{bmatrix}
    \frac{dz}{dw_1} & 0 & \cdots & 0 \\
    0 & \frac{dz}{dw_2} & \cdots & 0 \\
    \vdots & \vdots & \ddots & \vdots \\
    0 & 0 & \cdots & \frac{dz}{dw_n}
\end{bmatrix}
$$

$$
J_z(b) =
\begin{bmatrix}
    \frac{dz}{db_1} & 0 & \cdots & 0 \\
    0 & \frac{dz}{db_2} & \cdots & 0 \\
    \vdots & \vdots & \ddots & \vdots \\
    0 & 0 & \cdots & \frac{dz}{db_n}
\end{bmatrix}
$$

$$ dz = J_z(w) \, dw + J_z(b) \, db $$

$$ dz = \frac{dz}{dw} \, dw + \frac{dz}{db} \, db $$

$$ \text{Frobenius equation} $$

$$ 
\left \langle \frac{dF}{dw}, dw \right \rangle _F + 
\left \langle \frac{dF}{db}, db \right \rangle _F =
\left \langle \frac{dF}{dz}, dz \right \rangle _F
$$

$$
\left \langle \frac{dF}{dz}, \frac{dz}{dw} dw + 
\frac{dz}{db} db \right \rangle _F =
$$

$$
\left \langle \frac{dF}{dz}, \frac{dz}{dw} dw \right \rangle _F + 
\left \langle \frac{dF}{dz}, \frac{dz}{db} db \right \rangle _F =
$$

$$
\left \langle \Bigg( \frac{dz}{dw} \Bigg)^\top \frac{dF}{dz}, dw \right \rangle _F + 
\left \langle \Bigg( \frac{dz}{db} \Bigg)^\top \frac{dF}{dz}, db \right \rangle _F \implies
$$

$$
\begin{dcases}
    \frac{dF}{dw} = \Bigg( \frac{dz}{dw} \Bigg)^\top \frac{dF}{dz}
    \frac{dF}{db} = \Bigg( \frac{dz}{db} \Bigg)^\top \frac{dF}{dz}
\end{dcases}
$$

In [ ]:
import torch.autograd as autograd
import torch.nn as nn
import torch as tr

import import_ipynb
from approx import approx # type: ignore


def linear(x: tr.Tensor, w: tr.Tensor, b: tr.Tensor) -> tr.Tensor:
    """Linear function: z = x @ w + b."""

    return tr.addmm(b, x, w)


class LinearFunction(autograd.Function):
    """Custom autograd function for the `linear`."""

    @staticmethod
    def forward(ctx, x: tr.Tensor, w: tr.Tensor, b: tr.Tensor) -> tr.Tensor:
        ctx.save_for_backward(x)
        return linear(x, w, b)

    @staticmethod
    def backward(ctx, dF_dz: tr.Tensor) -> tuple[tr.Tensor, tr.Tensor, tr.Tensor]:
        (x, ) = ctx.saved_tensors

        dz_dw = x
        dF_dw = dz_dw.t() @ dF_dz
        dF_db = dF_dz.sum(dim=0)

        return (None, dF_dw, dF_db)

class Linear(nn.Module):
    """Custom module for the affine linear transformation."""

    def __init__(self, in_features: int, out_features: int, w: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        # `w` has shape (out_features, in_features) to be consistent with `nn.Linear`
        if w is None:
            self.weight = nn.Parameter(0.01 * tr.randn(out_features, in_features))
        else:
            self.weight = nn.Parameter(w.clone().detach().requires_grad_(True))

        if b is None:
            self.bias = nn.Parameter(0.01 * tr.randn(out_features, ))
        else:
            self.bias = nn.Parameter(b.clone().detach().requires_grad_(True))

    def forward(self, x):
        assert x.shape[1] == self.weight.shape[1]

        return LinearFunction.apply(x, self.weight.t(), self.bias)

In [ ]:
# Unit tests

def test_gradcheck(S: int, FI: int, FO: int) -> None:
    def fn(x, w, b):
        return LinearFunction.apply(x, w, b)

    x = tr.randn(S, FI, dtype=tr.float64, requires_grad=False)
    w = tr.randn(FI, FO, dtype=tr.float64, requires_grad=True)
    b = tr.randn(FO, dtype=tr.float64, requires_grad=True)

    assert autograd.gradcheck(fn, (x, w, b), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(S: int, FI: int, FO: int) -> None:
    nn_model = nn.Linear(FI, FO)
    dj_model = Linear(FI, FO, nn_model.weight, nn_model.bias)

    x = tr.randn(S, FI, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    z1 = dj_model(x1).sum()
    z1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    z2 = nn_model(x2).sum()
    z2.backward()

    assert x1 == approx(x2)
    assert dj_model.weight == approx(nn_model.weight)
    assert dj_model.weight.grad == approx(nn_model.weight.grad)
    assert dj_model.bias == approx(nn_model.bias)
    assert dj_model.bias.grad == approx(nn_model.bias.grad)


if __name__ == "__main__":
    test_gradcheck(1, 1, 1)
    test_gradcheck(10, 1, 1)
    test_gradcheck(10, 20, 1)
    test_gradcheck(10, 30, 30)

    test_compare(1, 1, 1)
    test_compare(10, 1, 1)
    test_compare(10, 20, 1)
    test_compare(10, 30, 30)
